## Prompt

> In an enumerated (or combinatorial) database, find molecules with lowest tanimoto similarity in relation to the query and with highest possible cosine similarity (espsim) at the same time. List them according to their price (highest to lowest).

This asks for a scaffold-hopping-style search: molecules that look different in 2D (low ECFP4/Tanimoto similarity to the query) but match electrostatically in 3D (high CHEESE ESP cosine similarity), against Enamine REAL. The DeepMedChem API only accepts `direction="maximize"` for similarity objectives/searches (verified live: requesting `"minimize"` is rejected server-side), so "lowest Tanimoto" cannot be requested directly - it has to be obtained and then selected for afterwards. This notebook does it two ways, against the same `enamine-real-v5a` database:

- **Part 1 (combinatorial):** a single `Selection` with two similarity objectives (`cheese.electrostatic` and `rdkit.ecfp4_tanimoto`), executed via the durable Run API so the server computes and reports both metrics per assembled candidate.
- **Part 2 (enumerated):** two independent `dmc.search()` calls against the precomputed CHEESE embedding index (`method="esp"` and `method="morgan"`), intersected by product id so each common candidate has both scores.

### Part 1 - Combinatorial (`Selection` + durable Run)

**Status: unverified.** The durable Runs backend is currently returning `503 credits_unavailable` on every run, including a trivial single-objective selection unrelated to this query (confirmed live, account has 999,997/1,000,000 credits remaining, so it isn't a quota issue). The code below is written against the documented `Selection`/`Run` API and validates successfully (`selections.validate()` returns `valid=True`), but has not been executed end-to-end, so the exact shape of `objective_components` per hit is a best-effort guess (a list of `{"metric_id": ..., "score": ...}` dicts, inferred from how other `Hit` fields are structured) and may need correcting once the backend recovers and this cell can actually run.

In [1]:
from deepmedchem import Client, Selection, Run

DATABASE = "enamine-real-v5a"
QUERY = "CC(=O)Oc1ccccc1C(=O)O"  # aspirin
TOP_N = 10

selection = (
    Selection.from_database(DATABASE)
    .reference("query", smiles=QUERY)
    .ranked()
    .maximize_similarity("cheese.electrostatic", reference="query")
    .maximize_similarity("rdkit.ecfp4_tanimoto", reference="query")
    .include("properties", "objective_components")
    .limit(50)
)
run = Run.selection(selection)

with Client() as dmc:
    dmc.runs.estimate(run)
    created = dmc.runs.create(run, idempotency_key="combinatorial-esp-low-tanimoto-001")
    terminal = dmc.runs.wait(created.id, timeout=180)
    if terminal.status not in {"completed", "completed_with_errors"}:
        raise RuntimeError(f"run {created.id} ended with status {terminal.status}")
    items = list(dmc.runs.iter_results(created.id, order="input"))
    if not items[0].ok:
        raise RuntimeError(f"selection failed: {items[0].error}")
    hits = items[0].result.get("results", [])


def component(hit, metric_id):
    for comp in hit.get("objective_components", []):
        if comp.get("metric_id") == metric_id:
            return comp.get("score")
    return None


# Both metrics were requested as "maximize" objectives (the API rejects "minimize" for
# similarity objectives), so this shortlist is already biased toward jointly-high
# candidates. We locally re-select for the lowest Tanimoto / highest ESP gap, then sort
# by price as requested.
scored = [
    (hit, component(hit, "cheese.electrostatic"), component(hit, "rdkit.ecfp4_tanimoto"))
    for hit in hits
]
scored = [row for row in scored if row[1] is not None and row[2] is not None]
scored.sort(key=lambda row: row[1] - row[2], reverse=True)
top = scored[:TOP_N]
top.sort(key=lambda row: row[0].get("price") or 0, reverse=True)

for hit, esp_score, tanimoto_score in top:
    print(
        f"price=${hit.get('price')}  esp={esp_score:.3f}  tanimoto={tanimoto_score:.3f}"
        f"  {hit['smiles']}"
    )

RuntimeError: selection failed: {'retryable': False, 'attempt_count': 1, 'code': 'invalid_item', 'run_id': 'run_5c06a66e2f7d42c5af2739cf5f', 'field': None, 'item_id': 'selection', 'retry_after_seconds': None, 'message': "503: {'code': 'credits_unavailable', 'message': 'Credit accounting is unavailable.'}"}

### Part 2 - Enumerated (`dmc.search()` against the CHEESE index)

This path is fully synchronous and does not touch the durable Runs backend, so it runs live below. Two independent nearest-neighbor searches (`method="esp"` and `method="morgan"`) are run against the same query and database, then intersected by `product_id` so each common molecule has both an ESP cosine score and a Tanimoto score. Within that intersection we rank by `esp - tanimoto` (the largest gap = most electrostatically similar relative to how fingerprint-different it is), then sort the top candidates by price, highest to lowest.

In [ ]:
from deepmedchem import Client

DATABASE = "enamine-real-v5a"
QUERY = "CC(=O)Oc1ccccc1C(=O)O"  # aspirin
SEARCH_LIMIT = 200
TOP_N = 10

with Client() as dmc:
    esp_result = dmc.search(QUERY, database=DATABASE, method="esp", limit=SEARCH_LIMIT)
    tanimoto_result = dmc.search(QUERY, database=DATABASE, method="morgan", limit=SEARCH_LIMIT)

esp_by_id = {hit.product_id: hit for hit in esp_result.hits}
tanimoto_by_id = {hit.product_id: hit for hit in tanimoto_result.hits}
common_ids = set(esp_by_id) & set(tanimoto_by_id)
print(f"esp hits={len(esp_result.hits)}  tanimoto hits={len(tanimoto_result.hits)}  overlap={len(common_ids)}")

candidates = [
    (esp_by_id[pid], esp_by_id[pid].score, tanimoto_by_id[pid].score)
    for pid in common_ids
]
candidates.sort(key=lambda row: row[1] - row[2], reverse=True)
top = candidates[:TOP_N]
top.sort(key=lambda row: row[0].price or 0, reverse=True)

for hit, esp_score, tanimoto_score in top:
    print(f"price=${hit.price}  esp={esp_score:.3f}  tanimoto={tanimoto_score:.3f}  {hit.smiles}")

esp hits=200  tanimoto hits=200  overlap=140
price=$245  esp=0.738  tanimoto=0.419  C=C(C)COc1ccccc1C(=O)Cc1ccccc1C(=O)O
price=$245  esp=0.723  tanimoto=0.425  CCOc1ccccc1C(=O)Cc1ccccc1C(=O)O
price=$245  esp=0.680  tanimoto=0.439  CC(=O)c1ccccc1OCC(=O)Cc1ccccc1C(=O)O
price=$245  esp=0.812  tanimoto=0.600  CC(=O)c1ccccc1OC(=O)O
price=$245  esp=0.635  tanimoto=0.425  COc1cccc(OC)c1C(=O)Cc1ccccc1C(=O)O
price=$163  esp=0.864  tanimoto=0.515  COC(=O)Oc1ccccc1C(C)=O
price=$163  esp=0.900  tanimoto=0.606  CC(C)(C)OC(=O)Oc1ccccc1C(=O)O
price=$163  esp=0.699  tanimoto=0.417  COC(=O)Cc1ccccc1C(=O)O
price=$163  esp=0.926  tanimoto=0.667  COC(=O)Oc1ccccc1C(=O)O
price=$163  esp=0.653  tanimoto=0.436  O=C(COc1ccccc1C(=O)O)Nc1ccccc1C(=O)O
